## PySpark DataFrames: Análisis del Censo

### **Introducción y objetivos del práctico**

Este *notebook* de PySpark vamos a trabajar con información del Censo 2023 de Uruguay.

Enlace de los *Microdatos Censo 2023 Anonimizados*: https://www.gub.uy/instituto-nacional-estadistica/politicas-y-gestion/microdatos-censo-2023-anonimizados


Conjunto que vamos a trabajar en esta clase: https://www5.ine.gub.uy/documents/CENSO%202023/Microdatos/personas_ext_26_02.rar


Cuestionario del censo: https://www.gub.uy/instituto-nacional-estadistica/sites/instituto-nacional-estadistica/files/2025-02/Cuestionario_censo2023%20%281%29.pdf


Diccionario de variables: https://www5.ine.gub.uy/documents/CENSO%202023/Microdatos/Diccionario%20de%20variables%202023.xlsx



Resumen del Proceso

- **Preparar el entorno:** Importar las bibliotecas necesarias y crear la sesión Spark.
- **Leer y preparar los datos:** Leer el archivo CSV, renombrar las columnas y transformar las características en un solo vector.
- **Dividir los datos:** Separar los datos en conjuntos de entrenamiento y prueba.
- **Entrenar el modelo:** Crear y entrenar el modelo de regresión lineal.
- **Evaluar el modelo:** Realizar predicciones y evaluar el rendimiento del modelo.

Con estos pasos, podrás construir y evaluar un modelo de regresión lineal en Spark.

In [1]:
# !pip install pyspark pandas

### **Configuración del entorno (SparkSession)**


In [2]:
import pandas as pd
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import when
from pyspark.sql.functions import avg, max, min, round, count, col
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import countDistinct
from pyspark.sql.functions import expr
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, RobustScaler
from pyspark.sql.functions import desc
from pyspark.sql.functions import corr
from pyspark.ml.evaluation import RegressionEvaluator

### **Carga y exploración inicial de datos**


In [3]:
spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .appName("MyApp") \
    .getOrCreate()

In [4]:
sdf = spark.read.csv('personas_ext_26_02.csv', header=True, inferSchema=True)

**Configuración opcional del entorno (máquina virtual)**

A continuación se incluyen algunos comandos de configuración para instalar y preparar el entorno de PySpark en una **máquina virtual o entorno local**.

**Importante:**  
Estos pasos **NO son necesarios en este entorno**, ya que ya está configurado.  
Se dejan como referencia por si trabajas fuera de este notebook.

---

### **Instalación de dependencias**

```python
#!apt-get update -y
#!apt-get install -y openjdk-17-jdk
#!pip install -q "pyspark>=3.6,<3.7"
#import os
#os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
#os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
#os.environ.pop("PYSPARK_SUBMIT_ARGS", None)
#os.environ.pop("SPARK_HOME", None)

In [5]:
print("Esquema del DataFrame de Spark:")
sdf.printSchema()

Esquema del DataFrame de Spark:
root
 |-- _c0: integer (nullable = true)
 |-- ID_CENSO: double (nullable = true)
 |-- DIRECCION_ID: string (nullable = true)
 |-- DEPARTAMENTO: integer (nullable = true)
 |-- LOCALIDAD: integer (nullable = true)
 |-- VIVID: string (nullable = true)
 |-- HOGID: string (nullable = true)
 |-- PERID: string (nullable = true)
 |-- REGION_4: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- MUNICIPIO_PAIS: string (nullable = true)
 |-- TIPO_MUNICIPIO_PAIS: string (nullable = true)
 |-- FUENTE_EXT: integer (nullable = true)
 |-- SIT_CALLE: integer (nullable = true)
 |-- CUESTIONARIO_COMPLETO: integer (nullable = true)
 |-- CUESTIONARIO_BASICO: integer (nullable = true)
 |-- RRAA: integer (nullable = true)
 |-- UNIVERSO: integer (nullable = true)
 |-- VIVVO00: integer (nullable = true)
 |-- PERPH02: integer (nullable = true)
 |-- PERNA01: integer (nullable = true)
 |-- PERNA01_TRAMO: string (nullable = true)
 |-- PERPA01: integer (nullable = tr

In [6]:
sdf.show(5)

+---+--------+------------+------------+---------+-----+-----+-----+--------+----+--------------+-------------------+----------+---------+---------------------+-------------------+----+--------+-------+-------+-------+-------------+-------+-------+---------+-------+---------+---------+---------+---------+---------+---------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+---------+---------+-------+-------+---------+-------+---------+---------+-------+---------+---------+-------+-------+-------+-------+-------+---------+---------+-------+---------+-------+-------+---------+-----------+---------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+---------+---------+-------+---------+---------+-------+---------+---------+-------+---------+-------+--------+--------+------------------------+-----------+----------+---------------+----------+---------------+------------+-----------------+-------+------------+
|_c

### **Selección y filtrado de datos**

In [7]:
sdf.select('MUNICIPIO_PAIS').show(20)

+------------------+
|    MUNICIPIO_PAIS|
+------------------+
|       Municipio E|
|       Municipio E|
|       Municipio D|
|       Municipio D|
|       Municipio D|
|     Sin Municipio|
|       Las Piedras|
|       Municipio A|
|       Municipio A|
|     Sin Municipio|
|          Progreso|
|Ciudad de la Costa|
|        Piriápolis|
|     Sin Municipio|
|       Municipio C|
|     Sin Municipio|
|       Municipio E|
|       Municipio E|
|       Municipio E|
|       Municipio F|
+------------------+
only showing top 20 rows


In [8]:
sdf.select('MUNICIPIO_PAIS').distinct().show()

+--------------------+
|      MUNICIPIO_PAIS|
+--------------------+
|            Progreso|
|              Garzón|
|           Rodríguez|
|               Sauce|
|      Joaquín Suarez|
|        San Bautista|
|              Casupá|
|               Pando|
|         Municipio B|
|         Municipio F|
|Nicolich Ciudad L...|
|    Ciudad del Plata|
|   Ombúes de Lavalle|
|         Bella Unión|
|Quebracho - Paysandu|
|          Santa Rosa|
|           Castillos|
|           Mariscala|
|          San Carlos|
|   Florencio Sánchez|
+--------------------+
only showing top 20 rows


In [9]:
sdf.filter(sdf['MUNICIPIO_PAIS'] == "Municipio E").show(20)

+------+--------+------------+------------+---------+-----+-----+-----+--------+----+--------------+-------------------+----------+---------+---------------------+-------------------+----+--------+-------+-------+-------+-------------+-------+-------+---------+-------+---------+---------+---------+---------+---------+---------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+---------+---------+-------+-------+---------+-------+---------+---------+-------+---------+---------+-------+-------+-------+-------+-------+---------+---------+-------+---------+-------+-------+---------+-----------+---------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+---------+---------+-------+---------+---------+-------+---------+---------+-------+---------+---------+--------+--------+------------------------+-----------+----------+---------------+----------+---------------+------------+-----------------+-------+------------

In [10]:
sdf.filter(sdf['DEPARTAMENTO'] < 4).show(20)

+---+--------+------------+------------+---------+-----+-----+-----+--------+----+------------------+-------------------+----------+---------+---------------------+-------------------+----+--------+-------+-------+-------+-------------+-------+-------+---------+-------+---------+---------+---------+---------+---------+---------+-------+-------+-------+-------+-------+-------+-------+------+-------+-------+-------+-------+-------+---------+---------+-------+-------+---------+-------+---------+---------+-------+---------+---------+-------+-------+-------+-------+-------+---------+---------+-------+---------+-------+-------+---------+-----------+---------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+---------+---------+-------+---------+---------+-------+---------+---------+-------+---------+-------+--------+--------+------------------------+-----------+----------+---------------+----------+---------------+------------+-----------------+-------+------------+

### **Transformación de columnas**

In [11]:
sdf_renombrado = sdf.select(
    col("ID_CENSO"),
    col("DEPARTAMENTO"),
    col("PERNA01").alias("Edad"),
    col("PERFM01").alias("Sexo"),
    col("MUNICIPIO_PAIS").alias("Municipio")
)

In [12]:
print("Esquema con columnas renombradas:")
sdf_renombrado.printSchema()

Esquema con columnas renombradas:
root
 |-- ID_CENSO: double (nullable = true)
 |-- DEPARTAMENTO: integer (nullable = true)
 |-- Edad: integer (nullable = true)
 |-- Sexo: integer (nullable = true)
 |-- Municipio: string (nullable = true)



In [13]:
print("Primeras filas con columnas renombradas:")
sdf_renombrado.show(5)


Primeras filas con columnas renombradas:
+--------+------------+----+----+-----------+
|ID_CENSO|DEPARTAMENTO|Edad|Sexo|  Municipio|
+--------+------------+----+----+-----------+
|     1.0|           1|  74|8888|Municipio E|
|     2.0|           1|  37|7777|Municipio E|
|     3.0|           1|  39|8888|Municipio D|
|     4.0|           1|  14|8888|Municipio D|
|     5.0|           1|  19|8888|Municipio D|
+--------+------------+----+----+-----------+
only showing top 5 rows


In [14]:
sdf_renombrado = sdf_renombrado.withColumn(
    "Grupo_Edad",
    when(col("Edad") < 18, "Menor")
    .when((col("Edad") >= 18) & (col("Edad") < 65), "Adulto")
    .when(col("Edad") >= 65, "Adulto Mayor")
    .otherwise("Sin Dato")
)

In [15]:
print("DataFrame con la nueva columna 'Grupo_Edad':")
sdf_renombrado.select("Edad", "Grupo_Edad").show(10)

DataFrame con la nueva columna 'Grupo_Edad':
+----+------------+
|Edad|  Grupo_Edad|
+----+------------+
|  74|Adulto Mayor|
|  37|      Adulto|
|  39|      Adulto|
|  14|       Menor|
|  19|      Adulto|
|5555|Adulto Mayor|
|  61|      Adulto|
|  30|      Adulto|
|  52|      Adulto|
|  52|      Adulto|
+----+------------+
only showing top 10 rows


In [16]:
sdf_renombrado.select("Edad").describe().show()

+-------+------------------+
|summary|              Edad|
+-------+------------------+
|  count|           3499451|
|   mean|1007.2870181637062|
| stddev| 2099.288864007407|
|    min|                 0|
|    max|              5555|
+-------+------------------+



## **Limpieza y Filtrado de Valores Atípicos/Especiales**

Sugerencia:

  - Revisar outliers (valores atípicos).

  - Revisar valores perdidos (missing data).

Estrategias de manejo de valores perdidos:
- imputación por el promedio (distribución normal), o por la mediana (distribución paranormal)

- imputación por distancia de mahalanobis: https://www.delftstack.com/es/howto/python/python-mahalanobis-distance/

- eliminar (tener cuidado cuando tienen pocos datos)

Criterio para imputar valores:
- Regla general (20%): Si una persona tiene más del 20% de sus respuestas perdidas en una escala o variable, sus datos deben ser considerados no fiables (posible recomendación eliminar ese caso según: Enders, 2022; Hughes et al., 2019;
Jakobsen et al., 2017)


*Referencias bibliograficas*

- Enders, C. K. (2022). Applied missing data analysis. Guilford Publications

- Hughes, R. A., Heron, J., Sterne, J. A. C., & Tilling, K. (2019). Accounting for missing data in
statistical analyses: multiple imputation is not always the answer. International Journal of
Epidemiology, 48(4), 1294-1304. https://doi.org/10.1093/ije/dyz032

- Jakobsen, J. C., Gluud, C., Wetterslev, J., & Winkel, P. (2017). When and how should multiple
imputation be used for handling missing data in randomised clinical trials – a practical guide
with flowcharts. BMC Medical Research Methodology, 17(1), 162.
https://doi.org/10.1186/s12874-017-0442-1

In [17]:
sdf_limpio = sdf_renombrado.filter(
    (col("Edad") < 120) & (col("Edad") > 0)
)

In [18]:
total_original = sdf.count()
total_limpio = sdf_limpio.count()

print(f"Total de registros original: {total_original}")
print(f"Total de registros después de filtrar edades atípicas: {total_limpio}")
print(f"Registros eliminados: {total_original - total_limpio}")



Total de registros original: 3499451
Total de registros después de filtrar edades atípicas: 2858099
Registros eliminados: 641352


### **Análisis descriptivo y agregaciones**

Cálculo de la edad promedio, máxima y mínima por municipio

In [19]:
sdf_limpio = sdf_limpio.groupBy("Municipio").agg(
    round(avg("Edad"), 2).alias("Edad_Promedio"),
    max("Edad").alias("Edad_Maxima"),
    min("Edad").alias("Edad_Minima"),
    count("*").alias("Total_Personas")
)

sdf_limpio.orderBy(col("Edad_Promedio").desc()).show(10)

+----------------+-------------+-----------+-----------+--------------+
|       Municipio|Edad_Promedio|Edad_Maxima|Edad_Minima|Total_Personas|
+----------------+-------------+-----------+-----------+--------------+
|          Montes|        49.99|         93|          1|           246|
|       Atlántida|        45.36|         96|          1|           625|
|    San Bautista|        45.23|         99|          1|           959|
|            Tala|        45.19|         99|          1|          3200|
|Colonia Valdense|        44.22|         96|          1|           372|
|      Santa Rosa|        43.99|        102|          1|          2791|
|     San Antonio|        43.88|         97|          1|          1303|
|            Soca|        43.86|        100|          1|           823|
|    Municipio CH|        43.51|        107|          1|        160496|
|     La Floresta|        43.07|         95|          1|           242|
+----------------+-------------+-----------+-----------+--------

PySpark SQL: Unificación de SQL con Spark

In [20]:
sdf_renombrado.createOrReplaceTempView("censo_personas")

print("Ejecutando consulta SQL: Conteo de personas por Municipio:")

consulta_sql = """
SELECT
    Municipio,
    COUNT(*) AS Total_Personas
FROM
    censo_personas
GROUP BY
    Municipio
ORDER BY
    Total_Personas DESC
"""

resultado_sql = spark.sql(consulta_sql).show()


Ejecutando consulta SQL: Conteo de personas por Municipio:
+------------------+--------------+
|         Municipio|Total_Personas|
+------------------+--------------+
|     Sin Municipio|        941464|
|       Municipio A|        203088|
|       Municipio F|        168683|
|       Municipio D|        166695|
|      Municipio CH|        161751|
|       Municipio B|        154588|
|       Municipio G|        150393|
|       Municipio E|        149054|
|       Municipio C|        145891|
|Ciudad de la Costa|        114633|
|         Maldonado|        106361|
|       Las Piedras|         61922|
|        San Carlos|         44476|
|             Pando|         39720|
|  Ciudad del Plata|         39108|
|           Salinas|         36343|
|    Barros Blancos|         33818|
|         Canelones|         31813|
|        18 de Mayo|         24161|
|        Piriápolis|         24011|
+------------------+--------------+
only showing top 20 rows


### **Regresión**

¿Existe relación entre la edad de la persona (PERNA01) y su nivel educativo alcanzado (NIVELEDU)?

PERNA01: edad (numérica continua).

NIVELEDU: nivel educativo (ordinal, pero codificado numéricamente).



![Modelos](https://th.bing.com/th/id/R.b6b5a3a73f568bd1f2b588feb93cd02a?rik=d0gt%2bmbw7VaQOw&riu=http%3a%2f%2fwww.favouriteblog.com%2fwp-content%2fuploads%2f2017%2f07%2fTypes-of-Learning.png&ehk=z%2btK2BpOxRNtB3u4Gis9bn3dIkKxuVwcqmENF9cGlx0%3d&risl=&pid=ImgRaw&r=0)


En esta práctica, continuaremos explorando los modelos supervisados, aquellos en los que el modelo aprende a partir de datos etiquetados. Estos modelos se dividen en:

- Regresión Lineal: Se usa cuando la variable objetivo es continua. Ejemplo: Predecir el precio de una vivienda en función de su superficie y ubicación.

-  Regresión Logística: Aunque su nombre sugiere "regresión", este modelo se usa para clasificación binaria o multiclase.
    - Calcula la probabilidad de que una observación pertenezca a una clase determinada.
    - Se usa en problemas como detección de spam, diagnóstico médico y predicción de abandono de clientes.

- Support Vector Machine (SVM): Es un modelo de clasificación que encuentra el hiperplano óptimo que separa las clases.
    - Funciona bien en problemas con datos complejos y espacios de alta dimensión.
    - Se usa en problemas como reconocimiento de imágenes y detección de fraudes.

Vamos a investigar las caracteristicas de la variable **NIVELEDU**


La siguiente tabla detalla la codificación para el máximo nivel educativo alcanzado (NIVELEDU) según la fuente de datos.

| Código (NIVELEDU) | Descripción (Máximo Nivel Educativo Alcanzado) |
| :---------------: | :--------------------------------------------- |
| 0                 | Menor de 4 años                                |
| 1                 | Preescolar                                     |
| 2                 | Primaria común o especial                      |
| 4                 | Educación media básica o Ciclo Básico          |
| 5                 | Educación media superior o Bachillerato        |
| 6                 | Capacitaciones o cursos de UTU                 |
| 7                 | Magisterio o profesorado                       |
| 8                 | Terciario no universitario                     |
| 9                 | Universidad o similar                          |
| 10                | Postgrado (Diploma/Maestría/Doctorado)         |
| 12                | Nunca asistió                                  |

In [21]:
sdf_reg = sdf.select(
    col("NIVELEDU"),
    col("PERNA01").alias("Edad")
    )

Ver el tipo con schema detallado

In [22]:
sdf_reg.printSchema()

root
 |-- NIVELEDU: integer (nullable = true)
 |-- Edad: integer (nullable = true)



Revisar estadísticos descriptivos

In [23]:
sdf_reg.describe().show()

+-------+------------------+------------------+
|summary|          NIVELEDU|              Edad|
+-------+------------------+------------------+
|  count|           3499451|           3499451|
|   mean|1142.3474959357911|1007.2870181637062|
| stddev| 3004.621562422023| 2099.288864007407|
|    min|                 0|                 0|
|    max|              9898|              5555|
+-------+------------------+------------------+



Contar cuántos valores distintos existen en la columna

In [24]:
sdf_reg.select(countDistinct('NIVELEDU')).show()

+------------------------+
|count(DISTINCT NIVELEDU)|
+------------------------+
|                      13|
+------------------------+



Contar frecuencia de cada valor

In [25]:
sdf_reg.groupBy('NIVELEDU').count().orderBy('NIVELEDU').show()


+--------+------+
|NIVELEDU| count|
+--------+------+
|       0|114990|
|       1| 70165|
|       2|797732|
|       4|734429|
|       5|585184|
|       6| 40120|
|       7| 97064|
|       8|103178|
|       9|382246|
|      10| 65083|
|      12| 69659|
|    8888|364142|
|    9898| 75459|
+--------+------+



Excluir los valores
|      12|
|    8888|
|    9898|

In [26]:
sdf_reg = sdf_reg.filter(~col("NIVELEDU").isin([12, 8888, 9898]))

Verificar

In [27]:
sdf_reg.select('NIVELEDU').distinct().orderBy('NIVELEDU').show()


+--------+
|NIVELEDU|
+--------+
|       0|
|       1|
|       2|
|       4|
|       5|
|       6|
|       7|
|       8|
|       9|
|      10|
+--------+



Vamos a investigar las caracteristicas de la variable **Edad**


Variable que tiene información de la edad en años.

Descripción:
- Edad en años cumplidos. Las personas residentes en localidades con menos de 10.000 habitantes no tienen dato en esta variable por secreto estadístico.

- Población estimada (excluyendo los residentes en localidades de menos de 10.000 habitantes)


In [28]:
sdf_reg.describe().show()

+-------+------------------+------------------+
|summary|          NIVELEDU|              Edad|
+-------+------------------+------------------+
|  count|           2990191|           2990191|
|   mean| 4.469915466938399|1014.0971362698905|
| stddev|2.5972651999286485|2104.6085382711058|
|    min|                 0|                 0|
|    max|                10|              5555|
+-------+------------------+------------------+



Contar cuántos valores distintos existen en la columna

In [29]:
sdf_reg.select(countDistinct('Edad')).show()

+--------------------+
|count(DISTINCT Edad)|
+--------------------+
|                 109|
+--------------------+



Contar frecuencia de cada valor

In [30]:
sdf_reg.groupBy('Edad').count().orderBy('Edad').show()

+----+-----+
|Edad|count|
+----+-----+
|   0|21442|
|   1|22631|
|   2|23640|
|   3|24964|
|   4|24931|
|   5|26995|
|   6|30446|
|   7|32407|
|   8|32797|
|   9|33158|
|  10|33222|
|  11|32709|
|  12|32831|
|  13|32820|
|  14|33034|
|  15|33227|
|  16|34152|
|  17|33213|
|  18|34237|
|  19|34007|
+----+-----+
only showing top 20 rows


Últimos 30 valores (mayores)

In [31]:
sdf_reg.groupBy('Edad').count().orderBy(desc('Edad')).show(30)

+----+------+
|Edad| count|
+----+------+
|5555|528705|
| 107|     5|
| 106|     3|
| 105|    12|
| 104|    22|
| 103|    33|
| 102|    46|
| 101|   100|
| 100|   177|
|  99|   237|
|  98|   373|
|  97|   534|
|  96|   766|
|  95|  1114|
|  94|  1425|
|  93|  1945|
|  92|  2571|
|  91|  3077|
|  90|  3844|
|  89|  4443|
|  88|  5045|
|  87|  5792|
|  86|  6789|
|  85|  7460|
|  84|  8547|
|  83|  9798|
|  82| 10413|
|  81| 10910|
|  80| 12009|
|  79| 13026|
+----+------+
only showing top 30 rows


Nos quedamos solo con las filas donde la edad es menor a 108.

In [32]:
sdf_reg = sdf_reg.filter(col("Edad") < 108)

Verificamos

In [33]:
sdf_reg.groupBy('Edad').count().orderBy(desc('Edad')).show(30)

+----+-----+
|Edad|count|
+----+-----+
| 107|    5|
| 106|    3|
| 105|   12|
| 104|   22|
| 103|   33|
| 102|   46|
| 101|  100|
| 100|  177|
|  99|  237|
|  98|  373|
|  97|  534|
|  96|  766|
|  95| 1114|
|  94| 1425|
|  93| 1945|
|  92| 2571|
|  91| 3077|
|  90| 3844|
|  89| 4443|
|  88| 5045|
|  87| 5792|
|  86| 6789|
|  85| 7460|
|  84| 8547|
|  83| 9798|
|  82|10413|
|  81|10910|
|  80|12009|
|  79|13026|
|  78|14514|
+----+-----+
only showing top 30 rows


Ver coeficientes de correlación

In [34]:
from scipy.stats import pearsonr

sdf_reg.select(corr("NIVELEDU", "Edad")).show()

+--------------------+
|corr(NIVELEDU, Edad)|
+--------------------+
| 0.16713076681476335|
+--------------------+



In [35]:
sdf_correlation = sdf_reg.select("NIVELEDU", "Edad").toPandas()

r, p = pearsonr(sdf_correlation["NIVELEDU"], sdf_correlation["Edad"])
print(f"r = {r:.4f}, p = {p:.6f}")

r = 0.1671, p = 0.000000


**Interpretación del coeficiente de correlación de Pearson**

El análisis de correlación de **Pearson** permite cuantificar la **fuerza y dirección** de la relación lineal entre dos variables numéricas.  
En este caso, estamos evaluando la relación entre:

- **NIVELEDU** → nivel educativo (variable dependiente)  
- **Edad** → indicador predictor (por ejemplo, una variable socioeconómica o demográfica del censo)

----

**r =** Es el coeficiente de correlación de Pearson. Toma valores entre -1 y +1.

**p =** Es el valor de significancia asociado al test estadístico. Permite evaluar si la correlación observada podría deberse al azar.


Cómo interpretar los resultados:
- Si r > 0, la relación es positiva: cuando aumenta Edad, también tiende a aumentar NIVELEDU.
- Si r < 0, la relación es negativa: cuando aumenta Edad, NIVELEDU tiende a disminuir.
- Cuanto más cerca de ±1, más fuerte es la relación lineal.
- Un valor cercano a 0 indica ausencia de relación lineal (aunque puede haber relación no lineal).

---
¿Cómo interpretar estos resultados?

Existe una correlación positiva débil pero significativa entre Edad y NIVELEDU.

Esto sugiere que, en promedio, a medida que aumenta Edad, el nivel educativo también tiende a aumentar ligeramente, aunque la relación no es fuerte.

### **AHORA SI: Regresión**

Objetivo

- Evaluar si la edad predice el nivel educativo.

En este ejemplo haremos:
- Variable dependiente (Y): NIVELEDU
- Variable independiente (X): PERNA01 (Edad)

## **Preparación de datos para regresión lineal (sin estandarización)**

En esta sección preparamos los datos para poder entrenar un modelo de regresión lineal en PySpark.

El objetivo es dejar las variables en el formato adecuado para que el modelo pueda procesarlas correctamente.

---

**1. Conversión de tipos de datos**

Convertimos las columnas **Edad** y **NIVELEDU** a tipo numérico (`double`).

Esto es necesario porque los modelos de Machine Learning en PySpark solo trabajan con variables numéricas.


In [36]:
sdf_reg = sdf_reg.withColumn("Edad", col("Edad").cast("double"))

sdf_reg = sdf_reg.withColumn("NIVELEDU", col("NIVELEDU").cast("double"))

**2. Eliminación de valores nulos**

Eliminamos los registros que tengan valores nulos en:

- **Edad**
- **NIVELEDU**

Esto es importante porque los modelos no pueden entrenarse correctamente con datos faltantes.

In [37]:
sdf_reg = sdf_reg.na.drop(subset=["Edad", "NIVELEDU"])

**3. Creación del vector de features**

Utilizamos **VectorAssembler** para combinar las variables de entrada en una sola columna llamada **features**.

Aunque en este caso solo usamos una variable (**Edad**), este paso es obligatorio porque los modelos en PySpark esperan las variables en formato vectorial.

In [38]:
assembler = VectorAssembler(inputCols=["Edad"], outputCol="features")

**4. Generación del dataset final**

Aplicamos la transformación y obtenemos un nuevo dataset:

- `assembled_no_scale`

Este dataset ya contiene la columna **features** lista para ser utilizada en el modelo de regresión.

In [39]:
assembled_no_scale = assembler.transform(sdf_reg)

**¿Por qué hacemos esto?**

PySpark ML no trabaja con columnas sueltas, sino con:

- Una columna de **features** (variables independientes)
- Una columna **label** (variable objetivo)

En este caso:

- **features → Edad**
- **label → NIVELEDU**

Con esto ya tenemos los datos preparados para entrenar el modelo sin aplicar escalado.

----------

**Dividir los datos en conjuntos de entrenamiento (80%) y prueba (20%)**


Para evaluar el rendimiento del modelo, se dividirá el dataset en dos subconjuntos:

🔹 Conjunto de entrenamiento (Train): Usado para entrenar el modelo.

🔹 Conjunto de prueba (Test): Usado para evaluar el rendimiento del modelo.

In [40]:
train_A, test_A = assembled_no_scale.randomSplit([0.8, 0.2], seed=42)

Mostrar la cantidad de registros en cada conjunto

In [41]:
print(f"Training Data Count: {train_A.count()}")

Training Data Count: 1969150


In [42]:
print(f"Test Data Count: {test_A.count()}")

Test Data Count: 492336


### Entrenar el Modelo de Regresión Lineal

En esta sección vamos a construir y entrenar un modelo de **regresión lineal**, cuyo objetivo es predecir el nivel educativo (**NIVELEDU**) a partir de las variables de entrada.

**1. Definición del modelo**

Creamos el modelo indicando:

- `featuresCol="features"` → columna que contiene las variables independientes (features)
- `labelCol="NIVELEDU"` → variable objetivo que queremos predecir

```python
lr_A = LinearRegression(featuresCol="features", labelCol="NIVELEDU")

In [43]:
lr_A = LinearRegression(featuresCol="features", labelCol="NIVELEDU")


**2. Entrenamiento del modelo**

Entrenamos el modelo utilizando el conjunto de datos de entrenamiento (`train_A`).

En este paso, el modelo aprende la relación entre las variables de entrada y la variable objetivo.

### **¿Qué está pasando aquí?**

El modelo ajusta una relación lineal entre las variables:

- Calcula los **coeficientes** (qué tanto influye cada variable)
- Calcula el **intercepto** (valor base)
- Minimiza el error entre valores reales y predichos

Esto permitirá luego hacer predicciones sobre nuevos datos.

In [44]:
lr_model_A = lr_A.fit(train_A)

### Evaluar el Modelo

Utilizaremos el conjunto de prueba para evaluar el modelo calculando el Error Cuadrático Medio (RMSE) y el coeficiente de determinación (R2).

**Métricas del modelo**

- **RMSE (Root Mean Squared Error):** mide el error promedio de las predicciones del modelo. Valores más bajos indican un mejor ajuste.  

- **R² (Coeficiente de determinación):** representa la proporción de la variabilidad de **NIVELEDU** que puede explicarse a partir de **Edad**. Un R² más cercano a 1 indica un mejor ajuste del modelo.

In [45]:
pred_A = lr_model_A.transform(test_A)
evaluator = RegressionEvaluator(labelCol="NIVELEDU", predictionCol="prediction")

rmse_A = evaluator.setMetricName("rmse").evaluate(pred_A)
r2_A   = evaluator.setMetricName("r2").evaluate(pred_A)

### Inspeccionar los Coeficientes del Modelo

Examinar los coeficientes e intercepto del modelo

In [46]:
print(f"RMSE (test): {rmse_A:.6f}")
print(f"R²   (test): {r2_A:.6f}")
print(f"Intercepto: {lr_model_A.intercept:.6f}")
print("Coeficientes:", lr_model_A.coefficients)

RMSE (test): 2.589331
R²   (test): 0.027675
Intercepto: 3.838395
Coeficientes: [0.019276487064230758]


**Resultados del modelo de regresión lineal**

*Coeficientes del modelo*

| Variable | Coeficiente | Interpretación |
|-----------|-------------|----------------|
| Edad | 0.0193 | Por cada unidad que aumenta el valor de **Edad**, el nivel educativo promedio (**NIVELEDU**) aumenta en aproximadamente **0.019 puntos**, manteniendo constante el resto de variables. Esto indica una **relación positiva**: a mayor Edad, ligeramente mayor nivel educativo. |

-----

*Intercepto del modelo*

| Intercepto | Valor | Interpretación |
|-------------|--------|----------------|
| Intercepto | 3.84 | Cuando **Edad = 0**, el valor estimado del nivel educativo (**NIVELEDU**) es **3.84**. Este representa el valor base del modelo, es decir, el punto donde la recta de regresión corta el eje Y. |

----

**Interpretación general**

El modelo muestra una **relación lineal positiva y débil** entre **Edad** y **NIVELEDU**: a medida que **Edad** aumenta, el nivel educativo promedio también tiende a incrementarse ligeramente.  

El **R² ≈ 0.0277** indica que el modelo explica solo una **muy pequeña proporción de la variabilidad** del nivel educativo.  

Esto significa que, aunque existe una tendencia positiva, **la variable Edad por sí sola no es suficiente para explicar el nivel educativo**, por lo que sería necesario incorporar más variables para mejorar el modelo.



Si el rendimiento del modelo no cumple con sus expectativas, puede probar las siguientes estrategias para mejorarlo:

- **Escalado de funciones:** estandarice o normalice las funciones de entrada para garantizar que estén en la misma escala.

- **Ajuste de hiperparámetros:** ajuste los hiperparámetros del modelo, como la intensidad de la regularización o el recuento de iteraciones.

---
---

---

---

## **Parte II: Modelo B – Regresión con RobustScaler**

En esta sección vamos a construir un segundo modelo aplicando **escalado de variables** mediante **RobustScaler**.

El objetivo es evaluar si el escalado mejora el rendimiento del modelo, especialmente en presencia de **valores atípicos (outliers)**.


**¿Por qué hacemos esto?**

El escalado permite:

- Evitar que variables con mayor escala dominen el modelo  
- Mejorar la estabilidad numérica  
- Reducir el impacto de valores extremos  



**1. Generación del vector de features sin escalar**

Primero creamos un vector de entrada con la variable **Edad**, sin aplicar ningún tipo de transformación.

Esto nos permite tener una versión "cruda" de los datos antes del escalado.


In [47]:
assembler_raw = VectorAssembler(inputCols=["Edad"], outputCol="raw_features")
assembled_raw = assembler_raw.transform(sdf_reg)

**2. División de datos (antes del escalado)**

Realizamos el **split en train y test antes de escalar**:

- 80% → entrenamiento  
- 20% → test  

Esto es muy importante para evitar **data leakage**.

El modelo de escalado debe aprender únicamente del conjunto de entrenamiento.

In [48]:
train_raw, test_raw = assembled_raw.randomSplit([0.8, 0.2], seed=42)

In [49]:
print(f"Training Data Count: {train_raw.count()}")
print(f"Test Data Count: {test_raw.count()}")

Training Data Count: 1969150
Test Data Count: 492336


**3. Aplicación de RobustScaler**

Utilizamos **RobustScaler**, que:

- Es menos sensible a outliers que otros escaladores
- Usa la **mediana** y el **rango intercuartílico (IQR)**

Importante:

- Se ajusta (**fit**) SOLO con los datos de entrenamiento  
- Luego se aplica (**transform**) tanto a train como a test  


In [50]:
scaler = RobustScaler(inputCol="raw_features", outputCol="features")
scaler_model = scaler.fit(train_raw)

train_B = scaler_model.transform(train_raw)
test_B  = scaler_model.transform(test_raw)

**4. Transformación de los datos**

Después del escalado:

- `train_B` → datos de entrenamiento escalados  
- `test_B` → datos de test escalados  

Ahora ambos conjuntos están listos para entrenar el modelo de regresión.

In [51]:
lr_B = LinearRegression(featuresCol="features", labelCol="NIVELEDU")
lr_model_B = lr_B.fit(train_B)

pred_B = lr_model_B.transform(test_B)

rmse_B = evaluator.setMetricName("rmse").evaluate(pred_B)
r2_B   = evaluator.setMetricName("r2").evaluate(pred_B)

Resultados MODELO B (RobustScaler)

In [52]:
print(f"RMSE (test, escalado): {rmse_B:.6f}")
print(f"R²   (test, escalado): {r2_B:.6f}")
print(f"Intercepto: {lr_model_B.intercept:.6f}")
print("Coeficientes:", lr_model_B.coefficients)

RMSE (test, escalado): 2.589331
R²   (test, escalado): 0.027675
Intercepto: 3.838395
Coeficientes: [0.6939535343130452]


Comparativa rápida

# **Interpretación**

Probamos dos versiones del mismo modelo de regresión lineal:

A) Sin estandarizar los datos.  
B) Con los datos escalados mediante RobustScaler, una técnica que reduce la influencia de valores atípicos.

El objetivo es ver si la estandarización mejora el rendimiento del modelo.

----

**RMSE (Error cuadrático medio):** Mide cuánto se alejan las predicciones de los valores reales.

- En ambos casos, el error es exactamente el mismo (**2.5893**) → no hubo mejora.

**R² (Coeficiente de determinación):** Indica qué proporción de la variabilidad de la variable dependiente explica el modelo.

- El valor es aproximadamente **0.0277** en ambos casos, lo que significa que el modelo explica solo alrededor del **2.77%** de la variabilidad.

Esto sugiere que el modelo tiene una **capacidad predictiva muy baja**, probablemente porque la variable utilizada no es un buen predictor o la relación no es lo suficientemente fuerte.

---

En este análisis, los resultados del modelo con y sin estandarización (RobustScaler) fueron **idénticos tanto en RMSE como en R²**.

Esto indica que la escala de las variables **no afecta el desempeño del modelo**, algo esperable en regresión lineal cuando se utiliza una sola variable.

Vamos a probar de nuevo la correlación

In [54]:
spark.stop()

Referencias para profundizar en este campo:


[Pyspark Tutorial: Getting Started with Pyspark](https://www.datacamp.com/tutorial/pyspark-tutorial-getting-started-with-pyspark)

[Linear Regression in PySpark](https://medium.com/@roshmitadey/a-comprehensive-guide-to-linear-regression-in-pyspark-810fdaf5c17c)

[Regresión lineal de PySpark: cómo crear y evaluar modelos de regresión lineal utilizando PySpark MLlib](https://www.machinelearningplus.com/pyspark/pyspark-linear-regression/?utm_content=cmp-true)










Regresión lineal: https://cienciadedatos.net/documentos/py10-regresion-lineal-python.html

Outliers: https://www.delftstack.com/es/howto/python/outlier-detection-python/
